In [ ]:
from dataclasses import dataclass, field
import numpy as np
import matplotlib.pyplot as plt

%matplotlib widget

In [ ]:
@dataclass
class MotorSteps:
    az_range_deg : np.ndarray
    el_range_deg : np.ndarray
    el_first : bool
    start : dict[str, int] = field(default_factory=lambda: {"az": 0, "el": 0})  # start pos in steps
    up_delay_us : dict[str, int] = field(default_factory=lambda: {"az": 2400, "el": 2400})
    dn_delay_us : dict[str, int] = field(default_factory=lambda: {"az": 300, "el": 600})
    slowdown_factor : int = 2
    slow_zone : int = 100  # in steps
    step_angle_deg : float = 1.8
    microstep : int = 1
    gear_teeth : int = 113

    def __post_init__(self):
        self.extra_delay = {k: self.slowdown_factor * v for k, v in self.dn_delay_us.items()}

        if self.el_first:
            self.axis1 = "az"  # the slow axis
            self.axis2 = "el"  # fast axis / inner loop
            self.ax1_rng = az_range_deg.copy()
            self.ax2_rng = el_range_deg.copy()
        else:
            self.axis1 = "el"
            self.axis2 = "az"
            self.ax1_rng = el_range_deg.copy()
            self.ax2_rng = az_range_deg.copy()

    def steps2deg(self, steps):
        s = steps / self.microstep / self.gear_teeth
        deg = s * self.step_angle_deg
        return float(deg)
    
    def deg_to_steps(self, degrees):
        s = degrees / self.step_angle_deg
        return int(s * self.microstep * self.gear_teeth)

    @property
    def steps(self):
        """
        Calculate the list of steps taken.

        Returns
        -------
        s : list
            Each element is a tuple of (axis, number of steps).

        """
        # steps per position
        # inner: full range every time
        nsteps_inner = self.deg_to_steps(self.ax2_rng[-1] - self.ax2_rng[0])
        # outer: one position each time  (assumes here that each position has the same step increment -- an always valid assumption in our case)
        nsteps_outer = self.deg_to_steps(self.ax1_rng[1] - self.ax1_rng[0])
        # for each outer position, take nsteps_inner steps on inner axis and then nsteps_outer steps on outer axis to get to next outer pos
        s = [(self.axis1, nsteps_outer), (self.axis2, nsteps_inner)] * self.ax1_rng.size
        # first outer step is taken into account in separate stow calculation so remove it from here
        return s[1:]

    @property
    def positions(self):
        """
        Get the arrays of postions.

        Returns
        -------
        dict
            Key is axis name, value is array of positions.
        """
        a2, a1 = np.meshgrid(self.ax2_rng, self.ax1_rng)
        a1 = a1.ravel()
        a2[1::2] = a2[1::2, ::-1]  # reverse every a1 position
        a2 = a2.ravel()
        return {self.axis1: a1, self.axis2: a2}

    def stow_time(self):
        raise NotImplementedError

    def calc_time(self, include_stow=False):
        t = 0
        for ax, ns in self.steps:
            nslow = np.minimum(2*self.slow_zone, ns)  # number of slow steps
            t += ns * (self.dn_delay_us[ax] + self.up_delay_us[ax]) + nslow * self.extra_delay[ax]
        if include_stow:
            t += self.stow_time()
        return t / 1e6

## 10 steps (Thursday - Saturday)

In [ ]:
az_range_deg = np.linspace(-180.0, 180.0, 10)
el_range_deg = np.linspace(-180.0, 180.0, 10)

### Az first (Thursday, Friday)

In [ ]:
s = MotorSteps(az_range_deg, el_range_deg, el_first=False)
print(f"{s.calc_time() / 60} min")

plt.figure()
plt.plot(s.positions["el"], label="el")
plt.plot(s.positions["az"], label="az")
plt.legend(loc="upper left")
plt.xlabel("Step number")
plt.ylabel("Position [deg]")
plt.show()

### El first (Saturday?)

In [ ]:
s = MotorSteps(az_range_deg, el_range_deg, el_first=True)
print(f"{s.calc_time() / 60} min")

plt.figure()
plt.plot(s.positions["el"], label="el")
plt.plot(s.positions["az"], label="az")
plt.legend(loc="upper left")
plt.xlabel("Step number")
plt.ylabel("Position [deg]")
plt.show()

## 72 steps (Sunday)

In [ ]:
az_range_deg = np.linspace(-180.0, 180.0, 72, endpoint=False)
el_range_deg = np.linspace(-180.0, 180.0, 72, endpoint=False)

s = MotorSteps(az_range_deg, el_range_deg, el_first=True)
st = s.calc_time()
hrs = st // 3600
rem = st - 3600 * hrs
print(f"{st // 3600} hr, {rem / 60} min")

plt.figure()
plt.plot(s.positions["el"], label="el")
plt.plot(s.positions["az"], label="az")
plt.legend(loc="upper left")
plt.xlabel("Step number")
plt.ylabel("Position [deg]")
plt.show()